# ¿Lee bien una IA multimodal los gráficos y tablas cuando cambia la presentación?
**Proyecto de investigación · Inteligencia Artificial · Tema 07: IA multimodal**

## A. Identificación

| Campo | Respuesta |
|---|---|
| Curso y grupo | Inteligencia Artificial · Grupo XX |
| Integrantes | Santiago Guevara · Januar Mena |
| Tema | 07 · IA multimodal |
| Título del estudio | Robustez de Gemini al leer gráficos de barras y tablas ante cambios de presentación |
| Fechas | Inicio: [__] · Exposición: [__] |
| Funciones | **Santiago:** diseño experimental, código, ejecución y análisis. **Januar:** revisión de fuentes, verificación de respuestas de referencia y clasificación de fallos. Cada semana revisamos el trabajo del otro (rotación de revisión). |

**Motivación** *(Januar)*: ✍️ *Pendiente: 3–4 líneas sobre por qué importa que una IA lea bien gráficos (informes, decisiones, accesibilidad) y su relación con el curso (percepción → representación del conocimiento).*

## B. Pregunta, hipótesis y alcance

- **Pregunta:** ¿Gemini lee correctamente un valor de un gráfico de barras o de una tabla, y mantiene esa lectura cuando solo cambia la presentación (no los datos)?
- **Hipótesis:** en la versión clara acertará ≥ 80 %, pero el acierto caerá **≥ 15 puntos** con eje truncado o con distractores visuales, sobre todo en gráficos de barras (las tablas dependen de OCR, que es más estable).
- **Qué la contradiría:** caídas < 5 puntos en todas las variantes (el modelo es robusto) o acierto < 50 % ya en la versión clara (el problema no es la presentación sino la lectura básica).
- **Objetivo general:** medir si la lectura visual del modelo depende de la presentación.
- **Objetivos específicos:** (1) medir el acierto por variante; (2) medir consistencia y estabilidad entre repeticiones; (3) clasificar los tipos de fallo.
- **Alcance:** 1 modelo (Gemini Flash vía API gratuita), gráficos sintéticos con datos conocidos. **Fuera:** gráficos reales de internet, otros tipos de gráfico, comparación entre modelos.
- **Alternativa si falla el acceso:** misma prueba manual en la interfaz web de Gemini con 10 familias, registrando cada respuesta.

## C. Fuentes y marco conceptual

*(Januar: completar con 5 fuentes de [docs/fuentes_candidatas.md](docs/fuentes_candidatas.md); ≥ 3 académicas y ≥ 2 de 2025–2026.)*

**Estrategia de búsqueda:** Google Scholar y arXiv, consultas `chart question answering multimodal LLM`, `misleading visualizations MLLM`, `vision language model robustness charts`; 2021–2026; se priorizan artículos originales.

| ID | Referencia | Año | Tipo / revisión por pares | Aporte | Limitación |
|---|---|---|---|---|---|
| F1 | | | | | |
| F2 | | | | | |
| F3 | | | | | |
| F4 | | | | | |
| F5 | | | | | |

**Síntesis crítica** *(Januar)*: ✍️ *Pendiente.*

**Cómo funciona un modelo multimodal (en nuestras palabras):**

```
imagen ──► codificador visual (ViT) ──► "tokens visuales" ──► proyección ─┐
                                                                          ├──► modelo de lenguaje ──► "72"
pregunta (texto) ──────────────────────► tokens de texto ─────────────────┘
```
La imagen se corta en parches; un *Vision Transformer* convierte cada parche en un vector. Una capa de proyección pone esos vectores en el mismo "idioma" que las palabras (**alineación texto–imagen**, idea popularizada por CLIP), y el modelo de lenguaje genera la respuesta mirando ambos. El modelo **no mide píxeles con regla**: estima. Leer una tabla depende sobre todo de reconocer texto (**OCR**); leer una barra exige **razonamiento espacial** (comparar la altura con el eje). Por eso esperamos más fallos en barras.

- **Datos:** imágenes sintéticas con valores conocidos · **Modelo:** Gemini Flash · **Herramientas:** matplotlib, API de Gemini · **Decisiones:** variantes, tolerancia ±3 · **Evaluación:** comparación automática con la referencia.

## D. Protocolo experimental

| Elemento | Decisión |
|---|---|
| Condición A (referencia) | Gráfico/tabla **clara** |
| Condición B (alternativas) | Mismos datos con: **escala** (barras: eje Y truncado; tablas: separador de miles y letra pequeña), **reordenada** (otro orden y colores), **distractor** (línea de meta, cuadros con otros números / columna Total) |
| Control | **datos_cambiados**: el valor preguntado cambia (±12–20 en barras, ±300–900 en tablas) y la referencia se recalcula |
| Factor que cambia | Solo la presentación (un factor por variante) |
| Se mantiene | Pregunta, prompt, modelo, temperatura, datos |
| Casos | 20 familias finales (12 barras + 8 tablas) × 5 imágenes; piloto separado de 6 familias |
| Referencia | Generada por código (`data/referencias.csv`); Januar verificó a mano 10 imágenes |
| Métrica principal | Acierto = valor leído dentro de **±3** (barras, escala 0–100) o **exacto** (tablas), con denominador |
| Secundarias | Error absoluto medio, estabilidad entre repeticiones, consistencia por familia, dirección correcta en el control, tiempo |
| Repeticiones | 3 por imagen (temperatura 1.0, valor por defecto) |
| Modelo y fecha | `gemini-flash-latest` → versión exacta registrada en `data/salidas.csv` · fecha: [__] |
| Exclusión | Errores de conexión se reintentan; respuestas sin número cuentan como fallo |

**Procedimiento:** `python src/generar_dataset.py` → `python src/consultar_modelo.py --fase final --reps 3` → `python src/evaluar.py --fase final`.

**Riesgos para la validez:** (1) las imágenes sintéticas son más limpias que las reales → no generalizamos a gráficos de internet; (2) la unidad independiente es la **familia** (20), no la consulta (300) → reportamos ambas y evitamos conclusiones finas; (3) el modelo puede cambiar de versión → guardamos todas las salidas.

## E. Piloto y bitácora

**Piloto (6 familias, 1 repetición):** [resultado y ajustes, p. ej. tolerancia, redacción del prompt]. **Protocolo fijado el:** [__].

| Semana | Integrante | Aporte verificable | Evidencia |
|---|---|---|---|
| 1 | Santiago | Repositorio, generador de imágenes, protocolo | commits en `src/` |
| 1 | Januar | Lectura de fuentes, tabla F1–F5 | PR `januar/fuentes` |
| 2 | Santiago | Script de consulta, piloto | `data/salidas.csv` |
| 2 | Januar | Verificación manual de 10 referencias | `docs/verificacion_referencias.md` |
| 3 | Santiago | Evaluación final, figura, análisis | `results/final/` |
| 3 | Januar | Clasificación de fallos | `data/etiquetas_fallos.csv` |

In [ ]:
# Configuración y ejemplo de una familia (mismos datos, 5 presentaciones)
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

refs = pd.read_csv("data/referencias.csv")
print(refs[refs.fase == "final"].groupby("tipo").familia.nunique().rename("familias"))

ej = refs[refs.familia == "F01"]
fig, ejes = plt.subplots(1, 5, figsize=(18, 3.2))
for ax, fila in zip(ejes, ej.itertuples()):
    ax.imshow(mpimg.imread(fila.archivo)); ax.axis("off")
    ax.set_title(f"{fila.variante}  (ref = {fila.respuesta_ref})", fontsize=10)
plt.show()
print("Pregunta:", ej.pregunta.iloc[0])

## F. Resultados

In [ ]:
res_dir = Path("results/final")
if (res_dir / "resumen_variantes.csv").exists():
    res = pd.read_csv(res_dir / "resumen_variantes.csv")
    res["acierto"] = res.aciertos.astype(str) + "/" + res.n.astype(str) + " (" + res.pct_acierto.astype(str) + " %)"
    orden = ["clara", "escala", "reordenada", "distractor", "datos_cambiados"]
    tabla = res.pivot(index="variante", columns="tipo", values="acierto").reindex(orden)
    display(tabla)
    display(res.pivot(index="variante", columns="tipo", values="error_abs_medio").reindex(orden).add_prefix("error medio · "))
    m = json.loads((res_dir / "resumen.json").read_text(encoding="utf-8"))
    for k in ["estabilidad_entre_repeticiones_pct", "consistencia_familia_pct", "control_direccion_correcta_pct",
              "tiempo_medio_s", "n_consultas", "modelo_version"]:
        print(f"{k}: {m.get(k)}")
else:
    print("Aún no hay resultados: ejecutar src/consultar_modelo.py y src/evaluar.py")

In [ ]:
from IPython.display import Image
fig_path = res_dir / "acierto_por_variante.png"
Image(filename=str(fig_path)) if fig_path.exists() else print("Figura pendiente")

**Interpretación de la figura:** [completar con los resultados: qué variante cae más, en qué tipo, cuántos puntos respecto a la clara].

## G. Análisis crítico

| Fallo | Evidencia | Causa posible | Cómo comprobarla | Mejora |
|---|---|---|---|---|
| 1 | [familia_variante, ref vs. respuesta] | | | |
| 2 | | | | |
| 3 | | | | |

*(Tipos de fallo clasificados por Januar en `data/etiquetas_fallos.csv`.)*

1. **¿Qué condición fue mejor y dónde?** [__]
2. **¿La evidencia respalda la hipótesis?** [__]
3. **¿Qué nos sorprendió?** [__]
4. **¿Coincide con la literatura?** [p. ej. Tonglet et al. 2025 reportan que los ejes truncados engañan a los MLLM]
5. **Explicaciones alternativas:** [__]
6. **Hasta dónde generalizamos:** solo gráficos sintéticos simples, un modelo, una fecha.
7. **Siguiente experimento:** pedir primero al modelo que extraiga la tabla de datos del gráfico y luego responder (estrategia propuesta por Tonglet et al. 2025).

**Conclusión:** [respuesta directa a la pregunta en 2–3 líneas, con números].

## H. Declaración de uso de IA y aportes

| Herramienta | Uso concreto | Parte afectada | Cómo lo verificamos |
|---|---|---|---|
| Claude (Anthropic), asistente de programación | Propuesta del diseño experimental, código en `src/`, borradores de redacción | B, D, F, código | Ejecutamos y revisamos el código; comprobamos imágenes contra `referencias.csv`; reescribimos y revisamos los textos |
| Gemini (Google) | **Objeto de estudio**, no fuente: sus respuestas son los datos del experimento | F, G | Registradas en `data/salidas.csv` con fecha y versión |

Las respuestas de los asistentes no se usan como referencias bibliográficas. Ambos integrantes entendemos y podemos explicar todo lo entregado.

**Reflexión individual** *(qué comprendí, qué hice, qué decisión defiendo, qué limitación veo)*:
- **Santiago:** ✍️
- **Januar:** ✍️

**Referencias:** *(Januar: formato APA, las 5 fuentes de la sección C.)*

## I. Exposición y verificación final

- **Hallazgo principal en una frase:** [__]
- **Caso de demostración:** una familia con acierto en la versión clara y fallo con eje truncado.
- **Intervenciones:** Januar: problema, fundamentos y conclusión · Santiago: método, resultados y demostración.

- [ ] Pregunta comprobable y 5 fuentes válidas
- [ ] Piloto separado de la evaluación final
- [ ] Tablas y figura coinciden con `data/salidas.csv`
- [ ] 3 fallos analizados y límites reconocidos
- [ ] Cuaderno ejecutado desde kernel limpio
- [ ] Uso de IA declarado y aportes registrados